# Checking consistency of `Sgrid` data

Import some packages.

In [1]:
from pathlib import Path
from typing import List

import numpy as np
import plotly.graph_objs as go

0D plotting functions

In [2]:
width = 1400
height= 1000

palette = ['#00b6ff',
           '#F6C324',
           '#DC4731',
           '#6A8023',
           '#DD819C',
           '#8BCA67',
           "#4269A2",
           '#FD7F20',
           '#A05987',
           '#000000',
           '#005BEA',
           '#A47551',
           '#746C70',
           '#970C10',
           '#055816',
           '#D7BA96',
           '#6E3562',
           '#C7CED6',
           '#BF8422',
           '#B5E5CF',
           '#523A28',
           '#15998E',
           '#C5B85E',
           ]

In [3]:
def plot_single_res_0D_fields(top: Path, fields : List[str], box = '0',
                             prop = 'rms'):
  fig = go.Figure()
  if not top.is_dir():
    print(f"This is not a directory:\n{str(top)}")
    return
  for field in fields:
    file_path = top / f"{field}_{prop}.{box}"
    if file_path.is_file():
      file_data = np.loadtxt(file_path)
      fig.add_trace(go.Scatter(x=file_data[:,0], y=file_data[:,1],
                          mode="lines+markers",
                          line=dict(width=1.25),
                          marker=dict(size=8, symbol="circle"),
                          name=field))
    else:
      print(f"Could not find: {file_path.name}. Skipping.")
  fig.update_layout(
    title=f"Box {box}",
    template='plotly_white',     # ← white background stylexaxis_title="x",
    colorway=palette,
    xaxis_title="Iteration",
    yaxis_title=f"Errors ({prop})",
    width=width,      # width in pixels
    height=height,     # height in pixels
    dragmode="pan",
    yaxis=dict(
      type="log",
      exponentformat="power",  # use powers of 10 notation
      showexponent="all"       # show exponent labels on all ticks
    ),
    font=dict(
      family="Garamond, Georgia, serif",
      size=20,
      color="black"
    ),
    legend=dict(
      orientation="h",     # horizontal
      yanchor="bottom",    # anchor legend to bottom of its boxline_color
      y=1.02,              # place just above the plot
      xanchor="center",    # anchor horizontally at center
      x=0.5                # center it
    )
  )

  # lock aspect ratio: 1 unit in x = 1 unit in y
  # fig.update_yaxes(scaleanchor="x", scaleratio=1)
  fig.show()

2D plotting function.

In [4]:
def _get_time_from_header(line):
  """
  Extract time from a line like:

      # "time = -40", k=5, Z=...

  Returns None if the line is not a time header.
  """

  line = line.strip()

  if not line.startswith("#"):
    return None

  if "time" not in line or "=" not in line:
    return None

  try:
    s = line.split("time", 1)[1]
    s = s.split("=", 1)[1]
    s = s.split(",", 1)[0]
    s = s.replace('"', '').strip()
    return float(s)
  except Exception:
    return None

########################################################################
def _load_time_surface(fp, time, time_tol=1e-12):
  """
  Load one time block from a structured 2D surface file.

  Important:
  Blank lines are used to detect rows of the 2D surface.

  Returns
  -------
  X, Y, Z : 2D arrays
  """

  rows = []
  current_row = []

  found_time = False
  in_requested_block = False

  with open(fp, "r") as f:
    for line in f:
      stripped = line.strip()

      header_time = _get_time_from_header(stripped)

      if header_time is not None:
        if in_requested_block:
          break

        if np.isclose(header_time, time, atol=time_tol, rtol=0):
          found_time = True
          in_requested_block = True
          rows = []
          current_row = []

        continue

      if not in_requested_block:
        continue

      if stripped == "":
        if current_row:
          rows.append(current_row)
          current_row = []
        continue

      parts = stripped.split()

      if len(parts) < 3:
        continue

      current_row.append((
        float(parts[0]),
        float(parts[1]),
        float(parts[2]),
      ))

  if current_row:
    rows.append(current_row)

  if not found_time:
    raise ValueError(f"Could not find time = {time} in {fp}")

  if not rows:
    raise ValueError(f"Found time = {time} in {fp}, but no data rows were found")

  row_lengths = [len(row) for row in rows]

  if len(set(row_lengths)) != 1:
    raise ValueError(
      f"Inconsistent row lengths in {fp} at time = {time}:\n"
      f"{row_lengths}"
    )

  ny = len(rows)
  nx = row_lengths[0]

  arr = np.asarray(rows, dtype=float)

  X = arr[:, :, 0]
  Y = arr[:, :, 1]
  Z = arr[:, :, 2]

  if X.shape != (ny, nx):
    raise RuntimeError("Internal reshape error")

  return X, Y, Z

########################################################################
def plot_2D_surface_from_files(top: Path,
                               field: str, boxes: List[str], plane='XY',
                               time=0, time_tol=1e-12,
                               cmap='Temps', opacity=1, width=1200, height=1200,
                               showcbar=False, drawlines=True,
                               licolor='black', drawevery=1, liwidth=1,
                               xtitle='x', ytitle='y', ztitle=None,
                               delim_color='black', delim_width=4):
  """
  Plot multiple 2D surface patches from time-evolved structured files.

  The files are expected to contain time blocks like:

      # "time = -40", k=5, Z=...
      x y z
      x y z
      ...

      x y z
      x y z
      ...

      # "time = -39.5", k=5, Z=...
      x y z
      x y z
      ...

  Blank lines inside each time block separate rows of the 2D surface.

  This function does NOT assume that the surface is a Cartesian product grid.
  It preserves the row/column structure in the file.
  """

  if not top.is_dir():
    print(f"This is not a directory:\n{str(top)}")
    return

  file_paths = []

  for box in boxes:
    file_path = top / f"{field}.{plane}{box}"

    if not file_path.is_file():
      print(f"Could not find: {file_path.name}. Skipping.")
      continue

    file_paths.append(file_path)

  if not file_paths:
    print("None of the files required for the plot were found. Quitting.")
    return

  patches = []

  zmins = []
  zmaxs = []

  for fp in file_paths:
    X, Y, Z = _load_time_surface(fp, time, time_tol=time_tol)

    patches.append((fp, X, Y, Z))

    zmins.append(np.nanmin(Z))
    zmaxs.append(np.nanmax(Z))

  zmin_global = min(zmins)
  zmax_global = max(zmaxs)

  traces = []

  first_surface = True

  drawevery = max(1, int(drawevery))

  for fp, X, Y, Z in patches:
    surface = go.Surface(
      x=X,
      y=Y,
      z=Z,
      colorscale=cmap,
      cmin=zmin_global,
      cmax=zmax_global,
      showscale=showcbar and first_surface,
      opacity=opacity,
      name=fp.name,
      showlegend=False
    )

    traces.append(surface)
    first_surface = False

    if drawlines:
      for j in range(0, X.shape[1], drawevery):
        traces.append(
          go.Scatter3d(
            x=X[:, j],
            y=Y[:, j],
            z=Z[:, j],
            mode='lines',
            line=dict(color=licolor, width=liwidth),
            showlegend=False
          )
        )

      for i in range(0, X.shape[0], drawevery):
        traces.append(
          go.Scatter3d(
            x=X[i, :],
            y=Y[i, :],
            z=Z[i, :],
            mode='lines',
            line=dict(color=licolor, width=liwidth),
            showlegend=False
          )
        )

    boundary_x = np.concatenate([
      X[0, :],
      X[1:, -1],
      X[-1, -2::-1],
      X[-2:0:-1, 0],
      X[0:1, 0],
    ])

    boundary_y = np.concatenate([
      Y[0, :],
      Y[1:, -1],
      Y[-1, -2::-1],
      Y[-2:0:-1, 0],
      Y[0:1, 0],
    ])

    boundary_z = np.concatenate([
      Z[0, :],
      Z[1:, -1],
      Z[-1, -2::-1],
      Z[-2:0:-1, 0],
      Z[0:1, 0],
    ])

    traces.append(
      go.Scatter3d(
        x=boundary_x,
        y=boundary_y,
        z=boundary_z,
        mode='lines',
        line=dict(color=delim_color, width=delim_width),
        showlegend=False
      )
    )

  fig = go.Figure(data=traces)

  vtitle = ztitle if ztitle else field

  fig.update_layout(
    template='plotly_white',
    width=width,
    height=height,
    title=f"{field}, boxes = {', '.join(boxes)}, iteration = {time}",
    scene=dict(
      xaxis_title=xtitle,
      yaxis_title=ytitle,
      zaxis=dict(
        title=vtitle,
        exponentformat="power",
        showexponent="all",
      ),
    ),
    font=dict(
      family="Garamond, Georgia, serif",
      size=20,
      color="black"
    ),
  )

  fig.show()

  return fig

### Outputs specs in the par file

The following pieces are needed in the par file, in addition to other desired outputs.

##### Single fluid runs:

```
0doutiter       = 1
0doutput        = DNSdata_Psi_Err DNSdata_Bx_Err DNSdata_alphaP_Err DNSdata_Sigma_Err ham momx normham normmomx
0doutputall     = yes
0doutput_VolumeIntegralJacobian = one
ADMvars_normalizedConstraints = yes

2doutiter       = 1
2doutput        = DNSdata_Sigmax
2doutputall     = yes
```

##### Two fluids runs:

```
0doutiter       = 1
0doutput        = DNSdataTwoFluid_Psi_Err DNSdataTwoFluid_Bx_Err DNSdataTwoFluid_alphaP_Err DNSdataTwoFluid_Sigma1_Err DNSdataTwoFluid_Sigma2_Err ham momx normham normmomx
0doutputall     = yes
0doutput_VolumeIntegralJacobian = one
ADMvars_normalizedConstraints = yes

2doutiter       = 1
2doutput        = DNSdataTwoFluid_Sigma1x DNSdataTwoFluid_Sigma2x
2doutputall     = yes
```

#### Initial flags

Set the flags for a whether run has two fluid and whether the component stars are identical.

In [5]:
two_fluids_data = True # set this to true if it is a two fluid run
plot_both_stars = False # set this to true if you want to plot fields for both stars

prefix = 'DNSdataTwoFluid' if two_fluids_data else 'DNSdata'

## Single resolution checks

It is sufficient to perform these checks with the final highest resolution data, but there is no harm in checking previous resolutions either.

Set the path of the top directory to checked and flag for whether it is two fluids data in the piece below.

In [6]:
target_dir = '/home/ananya/research/scratch/tests/sgrid/DNSTF_oz11oz21oz12oz220.005_b22_22_22_22'
# target_dir = '/home/ananya/research/scratch/tests/sgrid/SLy_DD_ec.45_b35_26_26_26'

top_path = Path(target_dir)


### Checks with 0D data

The idea is to check if the different error quantity norms are going down systematically with each iteration. 

In [7]:
quantities = [f'{prefix}_Psi_Err', f'{prefix}_Bx_Err', f'{prefix}_alphaP_Err', f'{prefix}_Sigma_Err', 'ham', 'momx', 'normham', 'normmomx']

##### Star 1

It is sufficient to check these in box 5 (which is one of the boxes fitted to the first star's surface) and box 0 (which is the first star's center). Box 5 is the more challenging of the two.

In [8]:
plot_single_res_0D_fields(top_path, quantities, box='5')

Could not find: DNSdataTwoFluid_Sigma_Err_rms.5. Skipping.


In [9]:
plot_single_res_0D_fields(top_path, quantities,)

Could not find: DNSdataTwoFluid_Sigma_Err_rms.0. Skipping.


##### Star 2

If the two stars are not identical, we have to check the same for the other star: box 13 is center and box 15 contains surface.

In [10]:
if plot_both_stars:
  plot_single_res_0D_fields(top_path, quantities, box='15')
  plot_single_res_0D_fields(top_path, quantities, box='13')

### Checks with 2D data

The derivatives of the velocity potential can have spikes if the data is not good. This is specially true for the $x$ and $z$ components, which theoretically should be zero, and hence are small in the actual data, leading to the possibility of spiky nature when errors are large. So, we need to ensure the profile is smooth. This is again most important in a box like 4, that is fitted to the star surface. But, it does not hurt to be more thorough. We also look primarily in the final iteration, 0, which is what goes in to the evolution.

##### Star 1

Plot box 4, for instance

In [11]:
if two_fluids_data:
  plot_2D_surface_from_files(top_path, 'DNSdataTwoFluid_Sigma1x', ['4'],)
  plot_2D_surface_from_files(top_path, 'DNSdataTwoFluid_Sigma2x', ['4'],)
else:
    plot_2D_surface_from_files(top_path, 'DNSdata_Sigmax', ['4'],)

For two fluids case, the second fluid is fully in box 0 for core configuration. So, we plot that.

In [12]:
if two_fluids_data:
  plot_2D_surface_from_files(top_path, 'DNSdataTwoFluid_Sigma2x', ['0'],)

Full star 1

In [13]:
s1_boxes = ['0','1','2','3','4']
if two_fluids_data:
  plot_2D_surface_from_files(top_path, 'DNSdataTwoFluid_Sigma1x', s1_boxes)
  plot_2D_surface_from_files(top_path, 'DNSdataTwoFluid_Sigma2x', s1_boxes)
else:
    plot_2D_surface_from_files(top_path, 'DNSdata_Sigmax', s1_boxes)

PLot even more if needed.

In [14]:
if False:
  s1_boxes = ['0','1','2','3','4','7','8','9','10']
  if two_fluids_data:
    plot_2D_surface_from_files(top_path, 'DNSdataTwoFluid_Sigma1x', s1_boxes)
    plot_2D_surface_from_files(top_path, 'DNSdataTwoFluid_Sigma2x', s1_boxes)
  else:
      plot_2D_surface_from_files(top_path, 'DNSdata_Sigmax', s1_boxes)

##### Star 2

PLot box 17.

In [15]:
if plot_both_stars:
  if two_fluids_data:
    plot_2D_surface_from_files(top_path, 'DNSdataTwoFluid_Sigma1x', ['17'],)
    plot_2D_surface_from_files(top_path, 'DNSdataTwoFluid_Sigma2x', ['17'],)
  else:
      plot_2D_surface_from_files(top_path, 'DNSdata_Sigmax', ['17'],)

For two fluid, plot box 13 for second fluid.

In [16]:
if plot_both_stars and two_fluids_data:
  plot_2D_surface_from_files(top_path, 'DNSdataTwoFluid_Sigma2x', ['13'],)

Full star 2.

In [17]:
if plot_both_stars:
  s2_boxes = ['13','14','15','16','17']
  if two_fluids_data:
    plot_2D_surface_from_files(top_path, 'DNSdataTwoFluid_Sigma1x', s2_boxes)
    plot_2D_surface_from_files(top_path, 'DNSdataTwoFluid_Sigma2x', s2_boxes)
  else:
      plot_2D_surface_from_files(top_path, 'DNSdata_Sigmax', s2_boxes)

More, if needed.

In [18]:
if False:
  s2_plus_boxes = ['13','14','15','16','17','20','21','22','23']
  if two_fluids_data:
    plot_2D_surface_from_files(top_path, 'DNSdataTwoFluid_Sigma1x', s2_plus_boxes)
    plot_2D_surface_from_files(top_path, 'DNSdataTwoFluid_Sigma2x', s2_plus_boxes)
  else:
      plot_2D_surface_from_files(top_path, 'DNSdata_Sigmax', s2_plus_boxes)